# Disproportionality Analysis — FEARS Pipeline Output
## PRR, ROR, and EBGM Signal Detection

Based on methodology from:
- **Evans et al. (2001)** — PRR
- **van Puijenbroek et al. (2002)** — ROR  
- **DuMouchel (1999)** — MGPS / EBGM

**Data:** Final pipeline output (adult + pediatric cohorts)  
**Subgroups:** Adult, Pediatric (overall), NICHD age bands (infancy, toddler, early/middle childhood, early/late adolescence)

In [ ]:
import polars as pl
import numpy as np
from scipy.stats import chi2_contingency
from scipy.special import gammaln
from scipy.stats import gamma as gamma_dist
from IPython.display import display, Markdown
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_ROOT = Path("../data")

adult = pl.read_parquet(DATA_ROOT / "adult" / "patient_report_reporter_drug_reaction_full_data.parquet")
pediatric = pl.read_parquet(DATA_ROOT / "pediatric" / "patient_report_reporter_drug_reaction_full_data.parquet")

print(f"Adult:      {adult.height:>10,} rows | {adult['safetyreportid'].n_unique():,} unique reports")
print(f"Pediatric:  {pediatric.height:>10,} rows | {pediatric['safetyreportid'].n_unique():,} unique reports")

## 1. Core Functions: PRR, ROR, EBGM

### The 2x2 Contingency Table

For each (drug, event) pair within a dataset:

|  | Target Event | Other Events | Total |
|--|--|--|--|
| **Target Drug** | a | b | a+b |
| **Other Drugs** | c | d | c+d |
| **Total** | a+c | b+d | N |

In [ ]:
# ============================================================
# Core disproportionality functions (optimized)
# ============================================================

def precompute_pair_counts(df: pl.DataFrame, drug_col: str, event_col: str) -> tuple:
    """Precompute all drug-event pair counts in one pass.
    
    Returns:
        pair_counts: DataFrame with (drug, event, count)
        drug_totals: dict {drug: total_reports}
        event_totals: dict {event: total_reports}
        N: total drug-event pairs
    """
    # Deduplicate to unique (report, drug, event)
    pairs = df.select(
        pl.col(drug_col).alias("drug"),
        pl.col(event_col).alias("event"),
    )
    
    N = pairs.height
    
    # Count per (drug, event) pair
    pair_counts = (
        pairs.group_by(["drug", "event"])
        .agg(pl.len().alias("count"))
    )
    
    # Drug marginals: total events per drug
    drug_totals = dict(
        pairs.group_by("drug")
        .agg(pl.len().alias("total"))
        .iter_rows()
    )
    
    # Event marginals: total drugs per event
    event_totals = dict(
        pairs.group_by("event")
        .agg(pl.len().alias("total"))
        .iter_rows()
    )
    
    # Build lookup dict for fast access
    pair_lookup = {}
    for row in pair_counts.iter_rows(named=True):
        pair_lookup[(row["drug"], row["event"])] = row["count"]
    
    return pair_lookup, drug_totals, event_totals, N


def get_2x2(pair_lookup, drug_totals, event_totals, N, target_drug, target_event):
    """Get 2x2 table values from precomputed counts (O(1) lookup)."""
    a = pair_lookup.get((target_drug, target_event), 0)
    drug_total = drug_totals.get(target_drug, 0)  # a + b
    event_total = event_totals.get(target_event, 0)  # a + c
    
    b = drug_total - a
    c = event_total - a
    d = N - a - b - c
    
    return a, b, c, d


def compute_prr(a, b, c, d):
    """Proportional Reporting Ratio with 95% CI and chi-squared.
    
    Signal: PRR >= 2 AND chi2 >= 4 AND a >= 3 (Evans 2001)
    """
    if a == 0 or c == 0 or (a + b) == 0 or (c + d) == 0:
        return {"PRR": np.nan, "PRR_lower": np.nan, "PRR_upper": np.nan,
                "chi2": np.nan, "PRR_signal": False}

    prr = (a / (a + b)) / (c / (c + d))
    se = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
    ci_lo = np.exp(np.log(prr) - 1.96 * se)
    ci_hi = np.exp(np.log(prr) + 1.96 * se)

    table = np.array([[a, b], [c, d]])
    chi2_val, _, _, _ = chi2_contingency(table, correction=False)

    signal = (prr >= 2) and (chi2_val >= 4) and (a >= 3)
    return {"PRR": round(prr, 2), "PRR_lower": round(ci_lo, 2),
            "PRR_upper": round(ci_hi, 2), "chi2": round(chi2_val, 2),
            "PRR_signal": signal}


def compute_ror(a, b, c, d):
    """Reporting Odds Ratio with 95% CI.
    
    Signal: ROR lower 95% CI > 1 AND a >= 3
    """
    if a == 0 or b == 0 or c == 0 or d == 0:
        return {"ROR": np.nan, "ROR_lower": np.nan, "ROR_upper": np.nan,
                "ROR_signal": False}

    ror = (a * d) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_lo = np.exp(np.log(ror) - 1.96 * se)
    ci_hi = np.exp(np.log(ror) + 1.96 * se)

    signal = (ci_lo > 1) and (a >= 3)
    return {"ROR": round(ror, 2), "ROR_lower": round(ci_lo, 2),
            "ROR_upper": round(ci_hi, 2), "ROR_signal": signal}


def compute_ebgm(a, E):
    """Simplified EBGM using fixed prior parameters.
    
    Uses typical FAERS prior: mixture of two Gamma distributions.
    Prior params from literature (DuMouchel 1999, openEBGM defaults).
    
    Signal: EBGM05 >= 2
    """
    # Default prior parameters (conservative, from openEBGM/PhViD literature)
    alpha_mix = 0.2     # mixing weight for signal component
    a1, b1 = 0.2, 0.1  # background component (most drug-event pairs)
    a2, b2 = 2.0, 4.0  # signal component (rare true signals)

    if E == 0 or a == 0:
        return {"EBGM": np.nan, "EBGM05": np.nan, "EBGM95": np.nan,
                "EBGM_signal": False}

    # Log posterior weights
    log_w1 = (np.log(alpha_mix) + a1 * np.log(b1) - (a1 + a) * np.log(b1 + E)
              + gammaln(a1 + a) - gammaln(a1))
    log_w2 = (np.log(1 - alpha_mix) + a2 * np.log(b2) - (a2 + a) * np.log(b2 + E)
              + gammaln(a2 + a) - gammaln(a2))

    log_max = max(log_w1, log_w2)
    Q = np.exp(log_w1 - log_max) / (np.exp(log_w1 - log_max) + np.exp(log_w2 - log_max))

    # Posterior mean = EBGM
    ebgm = Q * (a1 + a) / (b1 + E) + (1 - Q) * (a2 + a) / (b2 + E)

    # 5th and 95th percentiles from mixture posterior
    q1_05 = gamma_dist.ppf(0.05, a1 + a, scale=1/(b1 + E))
    q2_05 = gamma_dist.ppf(0.05, a2 + a, scale=1/(b2 + E))
    q1_95 = gamma_dist.ppf(0.95, a1 + a, scale=1/(b1 + E))
    q2_95 = gamma_dist.ppf(0.95, a2 + a, scale=1/(b2 + E))
    ebgm05 = Q * q1_05 + (1 - Q) * q2_05
    ebgm95 = Q * q1_95 + (1 - Q) * q2_95

    signal = ebgm05 >= 2
    return {"EBGM": round(ebgm, 2), "EBGM05": round(ebgm05, 2),
            "EBGM95": round(ebgm95, 2), "EBGM_signal": signal}


def compute_signals_batch(df, drug_col, event_col, pairs, label=""):
    """Compute PRR, ROR, EBGM for a list of (drug, event) pairs efficiently.
    
    Precomputes all pair counts once, then looks up each pair in O(1).
    """
    # Precompute once for the entire dataset
    pair_lookup, drug_totals, event_totals, N = precompute_pair_counts(df, drug_col, event_col)
    
    results = []
    for drug, event in pairs:
        a, b, c, d = get_2x2(pair_lookup, drug_totals, event_totals, N, drug, event)
        E = ((a + b) * (a + c)) / N if N > 0 else 0
        
        prr = compute_prr(a, b, c, d)
        ror = compute_ror(a, b, c, d)
        ebgm = compute_ebgm(a, E)
        
        results.append({
            "drug": drug, "event": event,
            "N_total": N, "a (cases)": a, "E (expected)": round(E, 2),
            **prr, **ror, **ebgm,
        })
    
    return pl.DataFrame(results)

print("Functions defined: precompute_pair_counts, get_2x2, compute_prr, compute_ror, compute_ebgm, compute_signals_batch")

## 2. Identify Top Drug-Reaction Pairs to Analyze

We pick the **top 30 most frequently reported drugs** in each cohort, then compute signals against their top reactions.

In [ ]:
# Get top drugs per cohort
def top_drug_event_pairs(df, n_drugs=30, n_events=5):
    """Find the top n_drugs, and for each drug its top n_events reactions."""
    top_drugs = (
        df.group_by("medicinal_product")
        .agg(pl.col("safetyreportid").n_unique().alias("n_reports"))
        .sort("n_reports", descending=True)
        .head(n_drugs)["medicinal_product"].to_list()
    )
    
    pairs = []
    for drug in top_drugs:
        drug_df = df.filter(pl.col("medicinal_product") == drug)
        top_events = (
            drug_df.group_by("reaction_meddrapt")
            .agg(pl.col("safetyreportid").n_unique().alias("n_reports"))
            .sort("n_reports", descending=True)
            .head(n_events)["reaction_meddrapt"].to_list()
        )
        for event in top_events:
            pairs.append((drug, event))
    return pairs

adult_pairs = top_drug_event_pairs(adult, n_drugs=30, n_events=3)
pediatric_pairs = top_drug_event_pairs(pediatric, n_drugs=30, n_events=3)

print(f"Adult pairs to analyze:     {len(adult_pairs)}")
print(f"Pediatric pairs to analyze: {len(pediatric_pairs)}")
print(f"\nExample adult pairs: {adult_pairs[:5]}")

## 3. Compute Signals — Adult vs Pediatric

In [ ]:
print("Computing Adult signals...")
adult_signals = compute_signals_batch(adult, "medicinal_product", "reaction_meddrapt", adult_pairs, label="Adult")

print("Computing Pediatric signals...")
ped_signals = compute_signals_batch(pediatric, "medicinal_product", "reaction_meddrapt", pediatric_pairs, label="Pediatric")

print(f"\nAdult signals:     {adult_signals.height} pairs")
print(f"Pediatric signals: {ped_signals.height} pairs")

## 4. Results Table — Top Signals

### Adult Cohort

In [ ]:
# Show top signals (all 3 methods agree)
def format_signal_table(signals_df):
    """Format and display top signals sorted by case count."""
    return (
        signals_df
        .select([
            "drug", "event", "a (cases)", "E (expected)",
            "PRR", "PRR_lower", "PRR_upper", "chi2", "PRR_signal",
            "ROR", "ROR_lower", "ROR_upper", "ROR_signal",
            "EBGM", "EBGM05", "EBGM95", "EBGM_signal",
        ])
        .sort("a (cases)", descending=True)
    )

display(Markdown("**Top 30 Adult Drug-Event Signals (sorted by case count):**"))
display(format_signal_table(adult_signals).head(30))

### Pediatric Cohort

In [ ]:
display(Markdown("**Top 30 Pediatric Drug-Event Signals (sorted by case count):**"))
display(format_signal_table(ped_signals).head(30))

## 5. Subgroup Analysis — NICHD Age Bands (Pediatric)

Recompute signals **within each NICHD age band** to see if signals differ by age group.

In [ ]:
subgroup_df = pl.DataFrame()  # default in case nichd column is missing

# Pediatric subgroup analysis by NICHD age bands
nichd_bands = ["infancy", "toddler", "early_childhood", "middle_childhood",
               "early_adolescence", "late_adolescence"]

# Use top 15 pediatric drug-event pairs for subgroup analysis
top_ped_pairs = pediatric_pairs[:15]

subgroup_dfs = []

if "nichd" in pediatric.columns:
    for band in nichd_bands:
        sub_df = pediatric.filter(pl.col("nichd") == band)
        if sub_df.height < 10:
            print(f"  {band}: only {sub_df.height} rows — skipped")
            continue
        print(f"  {band}: {sub_df.height:,} rows — computing signals...")
        band_signals = compute_signals_batch(sub_df, "medicinal_product", "reaction_meddrapt", top_ped_pairs, label=band)
        band_signals = band_signals.with_columns(
            pl.lit(band).alias("nichd_band"),
            pl.lit(sub_df.height).alias("subgroup_n"),
        )
        subgroup_dfs.append(band_signals)

    if subgroup_dfs:
        subgroup_df = pl.concat(subgroup_dfs)
        print(f"\nSubgroup results: {subgroup_df.height} rows across {len(subgroup_dfs)} age bands")
    else:
        subgroup_df = pl.DataFrame()
        print("No subgroups had enough data")
else:
    print("nichd column not available — skipping subgroup analysis")
    subgroup_df = pl.DataFrame()

In [ ]:
# Display subgroup comparison table
if subgroup_df.height > 0:
    comparison = (
        subgroup_df
        .select(["nichd_band", "subgroup_n", "drug", "event", "a (cases)",
                 "PRR", "PRR_signal", "ROR", "ROR_lower", "ROR_signal",
                 "EBGM", "EBGM05", "EBGM_signal"])
        .filter(pl.col("a (cases)") >= 3)  # Only show pairs with enough cases
        .sort(["drug", "event", "nichd_band"])
    )
    display(Markdown("**Pediatric Subgroup Signals (a >= 3, by NICHD band):**"))
    display(comparison)
else:
    print("No subgroup data to display")

## 6. Cross-Cohort Comparison — Same Drug-Event Pairs

Compare **the same drug-event pairs** across Adult, Pediatric overall, and NICHD subgroups.

In [ ]:
# Find drug-event pairs that exist in BOTH cohorts
common_drugs = set(adult["medicinal_product"].unique().to_list()) & set(pediatric["medicinal_product"].unique().to_list())
common_events = set(adult["reaction_meddrapt"].unique().to_list()) & set(pediatric["reaction_meddrapt"].unique().to_list())

# Pick top 20 common pairs by adult frequency
common_pairs = []
for drug, event in adult_pairs:
    if drug in common_drugs and event in common_events and len(common_pairs) < 20:
        common_pairs.append((drug, event))

print(f"Common drug-event pairs for cross-cohort comparison: {len(common_pairs)}")

# Compute for Adult
print("  Adult...")
cross_adult = compute_signals_batch(adult, "medicinal_product", "reaction_meddrapt", common_pairs, label="Adult")
cross_adult = cross_adult.with_columns(pl.lit("Adult").alias("cohort"))

# Compute for Pediatric overall
print("  Pediatric...")
cross_ped = compute_signals_batch(pediatric, "medicinal_product", "reaction_meddrapt", common_pairs, label="Pediatric")
cross_ped = cross_ped.with_columns(pl.lit("Pediatric").alias("cohort"))

cross_parts = [cross_adult, cross_ped]

# NICHD subgroups
if "nichd" in pediatric.columns:
    for band in nichd_bands:
        sub_df = pediatric.filter(pl.col("nichd") == band)
        if sub_df.height >= 10:
            print(f"  Ped-{band}...")
            cross_sub = compute_signals_batch(sub_df, "medicinal_product", "reaction_meddrapt", common_pairs, label=band)
            cross_sub = cross_sub.with_columns(pl.lit(f"Ped-{band}").alias("cohort"))
            cross_parts.append(cross_sub)

cross_df = pl.concat(cross_parts)
print(f"\nCross-cohort results: {cross_df.height} rows")

In [ ]:
# Display cross-cohort comparison
display(Markdown("### Cross-Cohort Signal Comparison"))
display(Markdown("Same drug-event pairs compared across Adult, Pediatric overall, and NICHD subgroups:"))

cross_table = (
    cross_df
    .select(["cohort", "drug", "event", "a (cases)", "E (expected)",
             "PRR", "PRR_signal", "ROR", "ROR_lower", "ROR_signal",
             "EBGM", "EBGM05", "EBGM_signal"])
    .sort(["drug", "event", "cohort"])
)
display(cross_table)

## 7. Signal Summary — Heatmap View

In [ ]:
import matplotlib.pyplot as plt

# Heatmap: PRR values across cohorts for common drug-event pairs
if cross_df.height > 0:
    # Create pivot: rows = drug-event, columns = cohort, values = PRR
    cross_with_label = cross_df.with_columns(
        (pl.col("drug") + " + " + pl.col("event")).alias("pair")
    )
    
    # Filter to pairs where at least one cohort has a signal
    has_signal = (
        cross_with_label
        .group_by("pair")
        .agg(pl.col("PRR_signal").any().alias("any_signal"))
        .filter(pl.col("any_signal"))["pair"].to_list()
    )
    
    signal_pairs = cross_with_label.filter(pl.col("pair").is_in(has_signal))
    
    if signal_pairs.height > 0:
        main_cohorts = ["Adult", "Pediatric"]
        pivot_data = (
            signal_pairs
            .filter(pl.col("cohort").is_in(main_cohorts))
            .select(["pair", "cohort", "PRR"])
            .pivot(on="cohort", index="pair", values="PRR")
            .sort("pair")
        )
        
        if pivot_data.height > 0:
            fig, ax = plt.subplots(figsize=(10, max(4, pivot_data.height * 0.4)))
            pairs_list = pivot_data["pair"].to_list()
            adult_prr = [v if v is not None else 0 for v in pivot_data.get_column("Adult").to_list()] if "Adult" in pivot_data.columns else []
            ped_prr = [v if v is not None else 0 for v in pivot_data.get_column("Pediatric").to_list()] if "Pediatric" in pivot_data.columns else []
            
            y = np.arange(len(pairs_list))
            h = 0.35
            if adult_prr:
                ax.barh(y - h/2, adult_prr, h, label="Adult PRR", color="#2196F3", alpha=0.8)
            if ped_prr:
                ax.barh(y + h/2, ped_prr, h, label="Pediatric PRR", color="#FF9800", alpha=0.8)
            ax.axvline(x=2, color="red", linestyle="--", alpha=0.5, label="PRR=2 threshold")
            ax.set_yticks(y)
            ax.set_yticklabels(pairs_list, fontsize=8)
            ax.set_xlabel("PRR")
            ax.set_title("PRR Comparison — Adult vs Pediatric (signal pairs only)")
            ax.legend()
            plt.tight_layout()
            plt.savefig("../notebook/prr_comparison.png", dpi=150, bbox_inches="tight")
            plt.show()
        else:
            print("No signal pairs to plot")
    else:
        print("No signal pairs found")
else:
    print("No cross-cohort data")

## 8. Export Results

In [ ]:
# Save all results to parquet for further analysis
output_dir = Path("../data/analysis")
output_dir.mkdir(parents=True, exist_ok=True)

adult_signals.write_parquet(output_dir / "adult_disproportionality_signals.parquet")
ped_signals.write_parquet(output_dir / "pediatric_disproportionality_signals.parquet")
cross_df.write_parquet(output_dir / "cross_cohort_comparison.parquet")
if subgroup_df.height > 0:
    subgroup_df.write_parquet(output_dir / "pediatric_nichd_subgroup_signals.parquet")

print(f"Results saved to {output_dir}/")
print(f"  - adult_disproportionality_signals.parquet ({adult_signals.height} rows)")
print(f"  - pediatric_disproportionality_signals.parquet ({ped_signals.height} rows)")
print(f"  - cross_cohort_comparison.parquet ({cross_df.height} rows)")
if subgroup_df.height > 0:
    print(f"  - pediatric_nichd_subgroup_signals.parquet ({subgroup_df.height} rows)")

---

## Methodology Notes

| Metric | Formula | Signal Threshold | Reference |
|--------|---------|-----------------|-----------|
| **PRR** | (a/(a+b)) / (c/(c+d)) | PRR >= 2, chi2 >= 4, a >= 3 | Evans et al., 2001 |
| **ROR** | (a*d) / (b*c) | ROR lower 95% CI > 1, a >= 3 | van Puijenbroek et al., 2002 |
| **EBGM** | Bayesian posterior mean (MGPS) | EBGM05 >= 2 | DuMouchel, 1999 |

**Subgroup analysis:** The same 2x2 table is reconstructed within each stratum (age band). Signals are recomputed independently per subgroup.

**EBGM prior:** This implementation uses fixed prior parameters from the literature. For production use, fit the prior globally using EM on all drug-event pairs.